# Stage 3.5 — Sanity-check the trained IQL policy

Loads the TorchScript policy (`data/policy.pt`) + the combined embedding
(`data/game_embeddings_matrix.npy` / `_index.pkl`) and prints the top-10
recommendations the policy produces from a cold-start state and four
distinct user profiles (RPG / FPS / strategy / indie).

**This is a human gate before Stage 4.** Eyeball whether the policy adapts
its recommendations to the user's history and whether cold-start picks are
plausible. The last code cell is left empty for ad-hoc queries.

The policy emits a continuous *action vector* (dim 1584); recommendations
are the catalog games whose embedding is closest (cosine) to that vector,
with the user's own played games filtered out — the same flow Stage 5
(`inference.py`) will use.


In [ ]:
import pickle
import numpy as np
import pandas as pd
import torch
from thefuzz import fuzz

E = np.load("../../data/game_embeddings_matrix.npy")
index_df = pd.read_pickle("../../data/game_embeddings_index.pkl")
names = index_df["name"].tolist()
name_to_row = dict(zip(index_df["name"].values, index_df["row_idx"].values))

# Load the TorchScript policy on CPU. The policy was traced on cuda:0 during
# training, so map_location="cpu" is required to run it without a GPU (this is
# also how the HuggingFace Space will load it).
policy = torch.jit.load("../../data/policy.pt", map_location="cpu")
policy.eval()

# Pre-normalize E once for cosine scoring.
E_norm = E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)

# Z = PCA-reduced action space (build_action_pca.py). The policy predicts in Z,
# so policy-mode matching uses Z_norm; profile (content-based) matching uses E_norm.
Z = np.load("../../data/game_actions_reduced.npy")
Z_norm = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)

print(f"Loaded policy + E {E.shape}, {len(names)} games")



In [ ]:
def resolve_name(query, min_ratio=80):
    """Fuzzy-resolve a (possibly user-typed) title to a canonical game name."""
    if query in name_to_row:
        return query
    best, best_score = None, -1
    q = query.lower()
    for n in names:
        s = fuzz.ratio(q, n.lower())
        if s > best_score:
            best, best_score = n, s
    return best if best_score >= min_ratio else None


def build_state(played_games):
    """Mean of E-vectors over recognized games; zero vector for cold start.

    Returns (state_vector, resolved_names). Mirrors the cold_start_state that
    Stage 6 will use, including fuzzy name resolution.
    """
    rows, resolved = [], []
    for g in played_games:
        canon = resolve_name(g)
        if canon is not None:
            rows.append(name_to_row[canon])
            resolved.append(canon)
    if not rows:
        return np.zeros(E.shape[1], dtype=np.float32), resolved
    return E[rows].mean(axis=0).astype(np.float32), resolved


In [ ]:
def top_k_recommendations(state, top_n=10, played=None, verbose=True):
    """Run the policy on `state`; return top-N games by cosine to the predicted action."""
    played = set(played or [])
    t = torch.from_numpy(np.asarray(state, dtype=np.float32))
    if t.ndim == 1:
        t = t[None, :]
    with torch.no_grad():
        action = policy(t).cpu().numpy().ravel()
    a_norm = action / max(float(np.linalg.norm(action)), 1e-12)
    sims = Z_norm @ a_norm
    results = []
    for idx in np.argsort(-sims):
        nm = names[idx]
        if nm in played:
            continue
        results.append((nm, float(sims[idx])))
        if len(results) == top_n:
            break
    if verbose:
        for i, (nm, sc) in enumerate(results, 1):
            print(f"{i:2d}. {sc:.4f}  {nm}")
    return results


def recommend_for(played_games, top_n=10):
    """Build a state from the played games and print the policy's top-N recs."""
    state, resolved = build_state(played_games)
    if resolved:
        print(f"History (resolved {len(resolved)}/{len(played_games)}): {resolved}")
    else:
        print("Cold start (empty history)")
    print("-" * 70)
    return top_k_recommendations(state, top_n=top_n, played=resolved)



## Cold start


In [ ]:
recommend_for([])


## RPG fan


In [ ]:
recommend_for(['The Witcher 3: Wild Hunt', 'Dark Souls III', 'The Elder Scrolls V: Skyrim'])


## FPS fan


In [ ]:
recommend_for(['DOOM Eternal', 'Halo Infinite', 'Titanfall 2'])


## Strategy fan


In [ ]:
recommend_for(["Sid Meier's Civilization VI", 'Total War: WARHAMMER'])


## Indie fan


In [ ]:
recommend_for(['Hollow Knight', 'Celeste', 'Stardew Valley'])


## Profile divergence diagnostic

How much does the policy actually adapt to the user's history? This computes
the pairwise Jaccard overlap of the top-10 recommendation sets across the
profiles above. Low overlap = the policy differentiates; high overlap (near 1)
= the policy is collapsing to roughly the same recommendations regardless of
input, which would be a red flag worth more training / tuning before Stage 4.


In [ ]:
_profiles = {
    'cold': build_state([])[0],
    'rpg': build_state(['The Witcher 3: Wild Hunt', 'Dark Souls III', 'The Elder Scrolls V: Skyrim'])[0],
    'fps': build_state(['DOOM Eternal', 'Halo Infinite', 'Titanfall 2'])[0],
    'strategy': build_state(["Sid Meier's Civilization VI", 'Total War: WARHAMMER'])[0],
    'indie': build_state(['Hollow Knight', 'Celeste', 'Stardew Valley'])[0],
}
_top10 = {k: {nm for nm, _ in top_k_recommendations(v, top_n=10, verbose=False)}
          for k, v in _profiles.items()}

_keys = list(_top10)
print(f"{'':10s}" + "".join(f"{k:>10s}" for k in _keys))
for a in _keys:
    row = f"{a:10s}"
    for b in _keys:
        j = len(_top10[a] & _top10[b]) / len(_top10[a] | _top10[b])
        row += f"{j:>10.2f}"
    print(row)
print("\n(1.00 on the diagonal; off-diagonal = Jaccard overlap of top-10 sets)")


## Stage 4 — policy reranking over a filtered candidate set

The cells above run the policy over the **whole catalog**, which surfaced a
recurring "default cluster" (Revenge of Arcade, WarGames, BattleZone…) in the
weaker profiles. Stage 4's candidate generator is meant to mask that: it first
narrows the catalog by the user's filters (year / platform / language) and
optionally reranks by similarity to the played games, then the policy reranks
*within that shortlist*.

`recommend_with_candidates` below previews the full Stage 5 flow. For one
profile (RPG) we decompose the contribution:

- **A — full catalog**: policy over all 26k games (Stage 3.5 behaviour).
- **B — filter only**: `candidates(filters)` (no profile rerank) → policy rerank. Isolates what the policy does over a filtered-but-unsorted set.
- **C — full pipeline**: `candidates(filters, played_games)` (filter + cosine profile rerank) → policy rerank. What Stage 5 ships.


In [ ]:
import sys
if ".." not in sys.path:
    sys.path.insert(0, "..")  # make the `app` package importable from source/scripts
from recommender.candidate_generator import candidates


def recommend_with_candidates(played_games, filters, top_n=10, use_profile_rerank=True, k=500, verbose=True):
    """Preview of Stage 5: candidate filter (+optional profile rerank) -> policy rerank -> drop played."""
    state, resolved = build_state(played_games)
    cg_played = resolved if use_profile_rerank else None
    cand_idx = candidates(filters, played_games=cg_played, k=k)

    t = torch.from_numpy(state[None, :].astype(np.float32))
    with torch.no_grad():
        action = policy(t).cpu().numpy().ravel()
    a = action / max(float(np.linalg.norm(action)), 1e-12)

    sims = Z_norm[cand_idx] @ a
    played_set = set(resolved)
    ranked = ((cand_idx[j], float(sims[j])) for j in np.argsort(-sims))
    out = [(names[i], sc) for i, sc in ranked if names[i] not in played_set][:top_n]

    if verbose:
        print(f"History: {resolved or 'cold start'}")
        print(f"Filters: {filters}  ->  {len(cand_idx)} candidates")
        print("-" * 70)
        for i, (nm, sc) in enumerate(out, 1):
            print(f"{i:2d}. {sc:.4f}  {nm}")
    return out




### RPG fan — A (full catalog) vs B (filter only) vs C (full pipeline)

Filter: released 2010+, on PC.


In [ ]:
_rpg = ["The Witcher 3: Wild Hunt", "Dark Souls III", "The Elder Scrolls V: Skyrim"]
_filt = {"year_min": 2010, "platforms": ["PC"]}

print("=== A. policy over FULL catalog ===")
recommend_for(_rpg)
print("\n=== B. filter only -> policy rerank ===")
recommend_with_candidates(_rpg, _filt, use_profile_rerank=False)
print("\n=== C. filter + profile rerank -> policy rerank (Stage 5) ===")
recommend_with_candidates(_rpg, _filt, use_profile_rerank=True, k=30)


### Indie fan — full pipeline

The noisiest profile in Stage 3.5. Filter: 2015+ (any platform).


In [ ]:
recommend_with_candidates(["Hollow Knight", "Celeste", "Stardew Valley"], {"year_min": 2015}, k=30)

### Filters actually bind — FPS fan, Nintendo Switch only, 2019+

Every recommendation should be a Switch game released in 2019 or later (or have unknown year/platform — the filter is lenient on missing metadata). The check below prints each pick's actual year + platforms.


In [ ]:
_recs = recommend_with_candidates(["DOOM Eternal", "Halo Infinite", "Titanfall 2"], {"year_min": 2019, "platforms": ["Nintendo Switch"]}, k=30)
print("\nFilter-compliance check:")
_rows = {nm: r for nm, r in zip(index_df["name"], index_df["row_idx"])}
for nm, _ in _recs:
    row = index_df.iloc[_rows[nm]]
    yr = row["release_year"]
    plats = [p for p in row["platforms"] if "switch" in str(p).lower()] or row["platforms"][:3]
    print(f"  {nm}: year={yr}, switch_platforms={plats}")

## Stage 5 — `inference.recommend` (full pipeline + metadata + compliance)

`recommend(state, filters, played_games, top_n)` is the function the app calls:
candidate filter → policy rerank → drop played → enrich with year/cover/description.

**Default mode is profile rerank** (`profile_prefilter=True`, `candidate_k=30`): the
candidate generator keeps the 30 games closest to the play history and the policy
reranks those — anchoring results to history and masking the policy's undertraining.
Cold start (no history) falls back to filter-only so the policy reranks the whole
filtered set. Pass `profile_prefilter=False` to always let the policy rank the full
filtered set (the "trust the policy" mode — see the Battlefield Hardline cell below).

`compliance_check` prints each recommendation's actual year / platforms / language
against the filter so you can confirm the filter bound — and see which picks pass
only via the *lenient-on-missing* rule.



In [ ]:
from recommender.inference import recommend


def show_recs(recs):
    for i, r in enumerate(recs, 1):
        yr = r["release_year"] if r["release_year"] is not None else "?"
        cov = "cover" if r["cover_url"] else "no-cover"
        print(f"{i:2d}. {r['score']:.4f}  {r['name']}  [{yr}, {cov}]")


def compliance_check(recs, filters):
    """Print each rec's year/platform/language vs the filter; flag lenient passes & violations."""
    ymin, ymax = filters.get("year_min"), filters.get("year_max")
    req_plat = [p.lower() for p in (filters.get("platforms") or [])]
    req_lang = (filters.get("language") or "").lower()
    print(f"Filter: {filters}")
    for r in recs:
        row = index_df.iloc[name_to_row[r["name"]]]
        yr, plats, langs = row["release_year"], list(row["platforms"]), list(row["language_supports"])
        notes = []
        if ymin is not None or ymax is not None:
            if np.isnan(yr):
                notes.append("year=NaN(lenient)")
            else:
                ok = (ymin is None or yr >= ymin) and (ymax is None or yr <= ymax)
                notes.append(f"year={int(yr)}{'' if ok else ' !VIOLATION'}")
        if req_plat:
            if not plats:
                notes.append("no-platforms(lenient)")
            else:
                hit = [p for p in plats if any((rp in str(p).lower()) or (str(p).lower() in rp) for rp in req_plat)]
                notes.append(f"plat={hit}" if hit else f"!PLATFORM VIOLATION {plats[:3]}")
        if req_lang:
            if not langs:
                notes.append("no-language(lenient)")
            else:
                ok = any(req_lang in str(l).lower() for l in langs)
                notes.append("lang-ok" if ok else "!LANG VIOLATION")
        print(f"  {r['name']}: {'; '.join(notes) if notes else '(no filter)'}")



### Scenario 1 — RPG fan · 2010+ · PC · default (profile rerank, k=30)



In [ ]:
_rpg = ["The Witcher 3: Wild Hunt", "Dark Souls III", "The Elder Scrolls V: Skyrim"]
_f = {"year_min": 2010, "platforms": ["PC"]}
_s, _resolved = build_state(_rpg)
_recs = recommend(_s, _f, played_games=_resolved, top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


### Scenario 2 — same query, `profile_prefilter=False` (trust the policy / filter-only)



In [ ]:
_recs = recommend(_s, _f, played_games=_resolved, top_n=5, profile_prefilter=False)
show_recs(_recs)
print()
compliance_check(_recs, _f)



### Scenario 3 — FPS fan · Nintendo Switch · 2019+ (does the year bind?)


In [ ]:
_fps = ["DOOM Eternal", "Halo Infinite", "Titanfall 2"]
_f = {"year_min": 2019, "platforms": ["Nintendo Switch"]}
_s2, _r2 = build_state(_fps)
_recs = recommend(_s2, _f, played_games=_r2, top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


### Scenario 4 — RPG fan · German-language filter (language leniency)


In [ ]:
_f = {"language": "German"}
_recs = recommend(_s, _f, played_games=_resolved, top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


### Scenario 5 — cold start · PlayStation 5 · 2020+


In [ ]:
_f = {"year_min": 2020, "platforms": ["PlayStation 5"]}
_recs = recommend(build_state([])[0], _f, played_games=[], top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


### Why "Battlefield Hardline: Getaway" disappears under the default profile rerank

It's a **PC game with no release date**, so it *passes* the `2010+ / PC` filter
(NaN year → lenient pass; PC is in its platform list). It was **not** sacked by
the filter. What drops it is the candidate generator's **profile-cosine rerank +
`candidate_k` cap (default 30)**: it ranks **593rd** in raw-embedding similarity to the
RPG play history (it's a Battlefield/FPS title), far outside the kept candidates.

The reason it looks great in filter-only mode is that the two stages score by
*different* criteria: the **policy's predicted action** points near it (cosine
≈0.75), even though its raw similarity to the literal play history is only ≈0.51.
Play-history cosine (candidate gen) and learned policy action (reranker) disagree —
that's the whole point of the `profile_prefilter` knob.



In [ ]:
_bf = "Battlefield Hardline: Getaway"
_row = name_to_row[_bf]
_f = {"year_min": 2010, "platforms": ["PC"]}
from recommender.candidate_generator import candidates as _cand
_filter_only = set(_cand(_f, played_games=None, k=None).tolist())
_profile_sorted = _cand(_f, played_games=_resolved, k=None).tolist()
print(f"{_bf}:")
print(f"  release_year = {index_df.iloc[_row]['release_year']}  (NaN -> lenient pass)")
print(f"  platforms    = {index_df.iloc[_row]['platforms']}")
print(f"  passes 2010+/PC filter?            {_row in _filter_only}")
print(f"  profile-rerank position (RPG)      {_profile_sorted.index(_row)} of {len(_profile_sorted)}  (default cap keeps 0..29)")
_profile = E[[name_to_row[g] for g in _resolved]].mean(0)
import torch as _t
with _t.no_grad():
    _act = policy(_t.from_numpy(_profile[None].astype(np.float32))).cpu().numpy().ravel()
_cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print(f"  cos(game, RPG play-history profile) = {_cos(E[_row], _profile):.4f}")
print(f"  cos(game, policy predicted action)  = {_cos(Z[_row], _act):.4f}")





## Ad-hoc queries

Drop any game list here to probe the policy.


In [ ]:
_games_ah = ["Marvel's Spider-Man", "Marvel's Spider-Man: Miles Morales", "Marvel's Spider-Man 2"]
_filter_ah = {"year_min": 2010, "platforms": ["Playstation 5"]}
_state_ah, _resolved_ah = build_state(_games_ah)
_recs_ah = recommend(_state_ah, _filter_ah, played_games=_resolved_ah, top_n=5, candidate_k=50)

show_recs(_recs_ah)
print()
compliance_check(_recs_ah, _filter_ah)